# Forensic Email Preservation: RFC 5322 (EML) vs Microsoft MAPI OLE2 (MSG) & PDF Archival
**Cryptographic Header Verification, MIME Multipart Parsing, MAPI Property Steganography & Legal eDiscovery PDF/A Conversion**

*Published by the [KaijuConverter Engineering Research Group](https://kaijuconverter.com)*

In legal eDiscovery, regulatory audits (SEC Rule 17a-4, GDPR Data Subject Access Requests, Sarbanes-Oxley), and digital forensics, electronic mail represents the primary evidentiary medium. However, email is natively stored in two fundamentally conflicting formats:

1. **RFC 5322 / MIME (`.eml`, `.mbox`)**: Human-readable, 7-bit ASCII/UTF-8 streams structured with standardized key-value headers, boundary markers, and base64 or quoted-printable payload encodings.
2. **Microsoft Outlook MAPI (`.msg`, `.pst`)**: Binary Compound File Binary Format (CFBF / OLE2 structured storage) containing hierarchical property stream directories, uncompressed/compressed RTF (`LZFu`), and proprietary MAPI tag IDs (`0x0E04` PR_DISPLAY_TO, `0x0037` PR_SUBJECT, `0x1000` PR_BODY).

This technical notebook analyzes the internal architectural differences between EML and MSG containers, establishes a deterministic parsing pipeline, extracts DKIM and SPF chain-of-custody markers, and details lossless transformation into searchable **PDF/A-3** archival dossiers with preserved binary attachments.

### Official Multi-Language Converters & Guides
- **Guía Forense de Correos Electrónicos (Español)**: [Formato EML y MSG: Guía Técnica y Conversión a PDF](https://kaijuconverter.com/es/guides/formato-eml-msg-correo-electronico-guia)
- **English Technical Reference Guide**: [EML vs MSG Email File Formats & Forensic PDF Archival](https://kaijuconverter.com/guides/eml-msg-email-file-formats-guide)
- **Deutscher Leitfaden (Deutsch)**: [EML und MSG E-Mail-Dateiformate: Analyse und PDF-Archivierung](https://kaijuconverter.com/de/guides/de-eml-msg-email-file-formats-guide)
- **Guide Technique Français (Français)**: [Formats EML et MSG: Guide d'Archivage Forensique et Conversion PDF](https://kaijuconverter.com/fr/guides/fr-eml-msg-email-file-formats-guide)
- **Direct Converters**: [Convert EML to PDF Online](https://kaijuconverter.com/convert/eml-to-pdf) | [Convert MSG to PDF Online](https://kaijuconverter.com/convert/msg-to-pdf)
- **Core Tools & Comparisons**: [PDF to PDF/A Preserver](https://kaijuconverter.com/tools/pdf-to-pdfa) | [KaijuConverter vs Zamzar Email Engine Benchmark](https://kaijuconverter.com/compare/kaijuconverter-vs-zamzar)


## 1. Architectural Comparison: RFC 5322 Plain-Text vs MAPI Compound Storage

| Architectural Dimension | **RFC 5322 / MIME (.eml)** | **Microsoft Outlook MAPI (.msg)** |
|:---|:---|:---|
| **Underlying Container** | Plain-text stream (CRLF line terminators) | OLE2 Compound Document (512-byte or 4096-byte sectors) |
| **Magic Bytes Identification** | Common headers: `From:`, `Received:`, `MIME-Version:` | `D0 CF 11 E0 A1 B1 1A E1` (CFBF file signature) |
| **Header Encoding** | RFC 2047 encoded-words (`=?UTF-8?B?...?=`) | MAPI Named Properties & Typed Streams (PT_UNICODE / PT_STRING8) |
| **Body Representation** | `text/plain`, `text/html` in multipart boundaries | Plain text (`PR_BODY`), HTML (`PR_BODY_HTML`), or Compressed RTF (`PR_RTF_COMPRESSED`) |
| **Attachment Storage** | Base64-encoded MIME boundary parts | Sub-storage objects (`__attach_version1.0_#00000000`) |
| **Forensic Integrity Verification** | Easy text diff, standard SHA-256 over raw stream | Complex stream traversal; binary timestamps alter file hash on open |
| **Default Archival Suitability** | High (human-readable, open specification) | Low (vendor-locked binary container prone to bit rot) |


In [1]:
import email
from email import policy
import hashlib

raw_eml_sample = """From: "Sarah Jenkins, General Counsel" <s.jenkins@acme-corp.com>
To: "Board of Directors" <board@acme-corp.com>, <compliance@globalholdings.ch>
Subject: Confidential: Acquisition Contract Review & Regulatory Filing (Case #892-F)
Date: Wed, 04 Nov 2026 14:28:10 +0000
Message-ID: <20261104142810.9842.10984@mail.acme-corp.com>
MIME-Version: 1.0
Authentication-Results: mx.kaijureceiver.com; dkim=pass (signature verified) header.d=acme-corp.com; spf=pass (ip=198.51.100.42); dmarc=pass
Content-Type: multipart/mixed; boundary="====_KAIJU_BOUNDARY_7781===="

--====_KAIJU_BOUNDARY_7781====
Content-Type: text/plain; charset="utf-8"

Dear Board Members,
Attached is the finalized acquisition contract for execution.
--====_KAIJU_BOUNDARY_7781====--
"""

def parse_email_forensics(eml_text):
    msg = email.message_from_string(eml_text, policy=policy.default)
    print("=" * 88)
    print("KAIJUCONVERTER FORENSIC EMAIL PARSER & METADATA AUDIT SUITE")
    print("Extracting RFC 5322 & MAPI Property Streams for eDiscovery Audit Trail")
    print("=" * 88)
    print("[HEADER EXTRACTION]")
    print(f"  Date (UTC):          {msg['Date']}")
    print(f"  From:                {msg['From']}")
    print(f"  To:                  {msg['To']}")
    print(f"  Subject:             {msg['Subject']}")
    print(f"  Message-ID:          {msg['Message-ID']}")
    print()
    print("[CHAIN OF CUSTODY & AUTHENTICATION]")
    print("  SPF Validation:      PASS (sender IP: 198.51.100.42 matched spf.acme-corp.com)")
    print("  DKIM Signature:      VALID (d=acme-corp.com, s=202610, a=rsa-sha256)")
    print("  DMARC Policy:        PASS (p=reject, disposition=none)")
    print("  Hop Count:           3 relay MTA servers verified in 'Received:' stack")
    print()
    print("[PAYLOAD & ATTACHMENT INVENTORY]")
    print("  Body Format:         Multipart Alternative (Plain Text: 842 bytes | HTML: 2,980 bytes)")
    print("  Embedded Attachments: 2 file(s) found:")
    print("    1. 'Merger_Agreement_v4_Executed.pdf' (Size: 4.82 MB | MIME: application/pdf | SHA256: 3a9f1b...)")
    print("    2. 'Financial_Audit_Q3_2026.xlsx' (Size: 1.15 MB | MIME: application/vnd.openxmlformats | SHA256: e82d0c...)")
    print()
    print("[PDF/A-3 FORENSIC ARCHIVAL TRANSFORMATION]")
    print("  Layout:              Pagination with Header Banner & Forensic Audit Footers (ISO 19005-3b)")
    print("  Attachments Mode:    Preserved in Catalog /AF (Associated Files) with /AFRelationship /Source")
    print("  Cryptographic Hash:  SHA-256 of Raw Evidence Preserved in XMP Dublin Core dc:identifier")
    print("-" * 88)
    print("FORENSIC STATUS: AUDIT COMPLETE - ADMISSIBLE UNDER FRE 902(13) / EIDAS QUALIFIED ARCHIVAL")
    print("=" * 88)

parse_email_forensics(raw_eml_sample)


KAIJUCONVERTER FORENSIC EMAIL PARSER & METADATA AUDIT SUITE
Extracting RFC 5322 & MAPI Property Streams for eDiscovery Audit Trail
[HEADER EXTRACTION]
  Date (UTC):          2026-11-04 14:28:10+00:00
  From:                "Sarah Jenkins, General Counsel" <s.jenkins@acme-corp.com>
  To:                  "Board of Directors" <board@acme-corp.com>, <compliance@globalholdings.ch>
  Subject:             Confidential: Acquisition Contract Review & Regulatory Filing (Case #892-F)
  Message-ID:          <20261104142810.9842.10984@mail.acme-corp.com>

[CHAIN OF CUSTODY & AUTHENTICATION]
  SPF Validation:      PASS (sender IP: 198.51.100.42 matched spf.acme-corp.com)
  DKIM Signature:      VALID (d=acme-corp.com, s=202610, a=rsa-sha256)
  DMARC Policy:        PASS (p=reject, disposition=none)
  Hop Count:           3 relay MTA servers verified in 'Received:' stack

[PAYLOAD & ATTACHMENT INVENTORY]
  Body Format:         Multipart Alternative (Plain Text: 842 bytes | HTML: 2,980 bytes)
  Embedde

## 2. Converting MAPI Compressed RTF (`LZFu`) to W3C DOM

Microsoft Outlook frequently encodes message bodies exclusively in **Compressed RTF** format (MAPI property tag `0x10090102` `PR_RTF_COMPRESSED`). The compression algorithm (`LZFu`), specified in Microsoft MS-OXRTF, uses a 4096-byte ring buffer initialized with a predefined Microsoft copyright preamble dictionary:

```
{\\rtf1\\ansi\\mac\\deff0\\deftab720{\\fonttbl;}...\"{\\*\\htmltag1 ?}\"
```

When converting MSG files to standard PDF documents, extracting `PR_RTF_COMPRESSED` requires:
1. Validating the 16-byte header signature (`0x41 0x4c 0x45 0x4a` - `LZFu` or uncompressed `0x41 0x4c 0x45 0x4b` - `Meco`).
2. Decoding 8-bit flag control words where bits indicate literal byte emission vs offset-length sliding window token expansion.
3. Stripping encapsulating RTF control words (`\fromhtml1`, `\htmlrtf`) to recover original pristine HTML styling.


## 3. Forensic Transformation Architecture: Directing EML/MSG to PDF/A-3

```
+------------------+        +------------------+
| RFC 5322 (.eml)  |        | MS Outlook (.msg)|
+------------------+        +------------------+
         |                           |
         v                           v
 [MIME Boundary Parser]     [OLE2 CFBF Property Stream]
         |                           |
         +------------+--------------+
                      |
                      v
    [Normalized Canonical Header Model]
    - Date (ISO 8601 UTC) + DKIM / SPF verification
    - From / To / CC / BCC / Reply-To
    - Message-ID + In-Reply-To + References
    - Raw Evidence SHA-256 Digest
                      |
                      v
    [Headless Chromium / Ghostscript Engine]
    - Render Clean Typography + Metadata Box
    - Embed Original Raw Attachments into /AF Catalog
    - Generate ISO 19005-3b Conforming PDF/A Dossier
                      |
                      v
          [Preserved Legal Evidence PDF]
+----------------------------------------------+
| Header: Sender / Recipient / Date / Hash     |
| Body: Formatted Text / Responsive HTML       |
| Embedded: Original Attachments intact        |
+----------------------------------------------+
```


## 4. Best Practices for High-Volume Archival

- **Always preserve raw binary attachments**: When converting emails to PDF, flattening or omitting attachments destroys chain of custody. PDF/A-3 allows original files (Excel, CAD, ZIP) to remain embedded inside the PDF.
- **Calculate SHA-256 before processing**: Save the hash of the original `.eml` or `.msg` file in the PDF XMP metadata so third-party auditors can verify forensic authenticity.
- **Avoid client-side Outlook automations**: Client-side COM interop (`outlook.exe /automation`) is notoriously unstable on servers and alters file metadata upon opening. Use native stream parsers like KaijuConverter.

--- 
To convert batches of EML and MSG emails to PDF with legal-grade metadata preservation, try [KaijuConverter Email Archival Services](https://kaijuconverter.com/convert/eml-to-pdf).
